In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("Exemple SparkSession") \
    .master("local[2]") \
    .getOrCreate()

print(spark.version)

df_events = spark.read.csv("data/events_2025-10-*", header=True)
df_events.show(5, truncate=False)

df_transactions = spark.read.csv("data/transactions_2025-10-*", header=True)
df_transactions.show(5, truncate=False)

df_join = df_transactions.join(df_events, how="left", on="transaction_id")
df_join.show(5, truncate=False)

df_agg = df_join.groupby(["load_time", "file_name"]).agg(
    F.count("*").alias("nb_total"),
    F.sum((F.when( (F.col("status") != "OK") or (F.col("status").isNull()), 1).otherwise(0)).alias("nb_invalid")
).withColumn(
    "error_rate", F.col("nb_invalid") / F.col("nb_total")
)
df_agg.show(3, truncate=False)

window_rank = Window.partitionBy("file_name").orderBy("load_time")
window_rolling = Window.partitionBy("file_name").orderBy("load_time").rowsBetween(-6, 0)

df_final = df_agg.withColumn(
    "rank", F.row_number().over(window_rank)
).withColumn(
    "rolling_error_rate_7d",
    F.round(F.avg("error_rate").over(window_rolling), 4)
)

df_final.show()



ModuleNotFoundError: No module named 'pyspark'

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# 🔧 Création de la SparkSession
spark = SparkSession.builder \
    .appName("AuditDaily") \
    .master("local[2]") \
    .getOrCreate()

# 1️⃣ Lecture des données (avec inférence de schéma possible pour + de robustesse)
df_events = spark.read.option("header", True).csv("data/events_2025-10-*")
df_transactions = spark.read.option("header", True).csv("data/transactions_2025-10-*")

# 2️⃣ Jointure entre transactions et events
df_join = df_transactions.join(df_events, on="transaction_id", how="left")

# 3️⃣ Extraction du jour depuis load_time
df_join = df_join.withColumn("load_date", F.to_date(F.col("load_time")))

# 4️⃣ Règle métier : détection des invalids
df_join = df_join.withColumn(
    "is_invalid",
    F.when(
        (F.col("status") != "OK") |
        (F.col("amount").isNull()) |
        (F.col("amount").cast("float") < 0),
        1
    ).otherwise(0)
)

# 5️⃣ Agrégation quotidienne par fichier
df_agg = df_join.groupBy("load_date", "file_name").agg(
    F.count("*").alias("nb_total"),
    F.sum("is_invalid").alias("nb_invalid")
).withColumn(
    "error_rate", 
    (F.col("nb_invalid") / F.col("nb_total")).cast("double")
)

# 6️⃣ Fenêtres analytiques : classement + rolling 7 jours
window_rank = Window.partitionBy("load_date").orderBy(F.col("error_rate").desc())
window_rolling = Window.partitionBy("file_name").orderBy("load_date").rowsBetween(-6, 0)

df_final = df_agg.withColumn(
    "rank", F.row_number().over(window_rank)
).withColumn(
    "rolling_error_rate_7d", 
    F.round(F.avg("error_rate").over(window_rolling), 4)
)

# 7️⃣ Résultat final
df_final.show(10, truncate=False)
